# Notebook 56 — Evaluación del RAG con MLflow

Evaluaremos dos versiones del prompt sobre exactamente las mismas preguntas:

1. **Capa A — determinística**: siempre corre primero, no usa judges y garantiza métricas
   aun cuando Free Edition aplique throttling.
2. **Capa B — LLM judges**: usa una muestra pequeña para Correctness, Groundedness y
   Retrieval Relevance.

Cada configuración produce su propio run de MLflow. Al final se guarda una tabla comparativa
en Delta (`agenteval_eval_summary`) para conservar un registro gobernado adicional.


## 1. Importar el agente

`%run` ejecuta las definiciones del notebook 55. Su guard evita registrar prompts o llamar
al LLM durante la importación.


In [0]:
# Estos widgets deben existir antes de %run para propagar una configuración
# distinta de los valores predeterminados al notebook 55.
for widget_name, default, label in [
    ("catalogo", "big_data_ii_2025", "1. Catálogo UC"),
    ("esquema", "spark_examples", "2. Schema UC"),
    ("volume", "agenteval_squadv2", "3. Volume UC"),
    ("endpoint_ai_search", "agenteval_ai_search", "4. Endpoint AI Search"),
    (
        "llm_endpoint",
        "databricks-meta-llama-3-1-8b-instruct",
        "5. Foundation Model endpoint",
    ),
]:
    try:
        dbutils.widgets.get(widget_name)
    except Exception:
        dbutils.widgets.text(widget_name, default, label)


In [0]:
%run ./55_RAG_Agente_MLflow


## 2. Parámetros de una evaluación pequeña y repetible


In [0]:
EXPECTED_RAG_AGENT_CONTRACT = "retriever_documents_v2"
actual_contract = globals().get("RAG_AGENT_CONTRACT_VERSION")
if actual_contract != EXPECTED_RAG_AGENT_CONTRACT:
    raise RuntimeError(
        "El notebook 56 importó una versión anterior de 55_RAG_Agente_MLflow. "
        f"Contrato esperado={EXPECTED_RAG_AGENT_CONTRACT!r}; observado={actual_contract!r}. "
        "Actualiza también el notebook 55, reinicia Python y ejecuta el 56 desde el inicio."
    )
print(f"Contrato del agente: {actual_contract} ✓")

dbutils.widgets.text("n_eval", "8", "1. Casos para Capa A")
dbutils.widgets.text("k", "3", "2. k del retriever")
dbutils.widgets.dropdown(
    "citation_policy",
    "intersect",
    ["intersect", "model", "all_retrieved", "top1"],
    "3. Política de citas",
)

N_EVAL = int(dbutils.widgets.get("n_eval"))
K_BASE = int(dbutils.widgets.get("k"))
POLICY_BASE = dbutils.widgets.get("citation_policy")

from pyspark.sql import functions as F

if N_EVAL < 1 or N_EVAL > 100:
    raise ValueError("n_eval debe estar entre 1 y 100.")
if K_BASE < 1:
    raise ValueError("k debe ser mayor o igual a 1.")

print(f"Experimento : {EXPERIMENT_PATH}")
print(f"N Capa A    : {N_EVAL}")
print(f"k           : {K_BASE}")
print(f"Política    : {POLICY_BASE}")
print(f"LLM         : {LLM_ENDPOINT}")


## 3. Dataset con la forma requerida por `mlflow.genai.evaluate`

Las claves de `inputs` deben coincidir con los parámetros de `predict_fn`. Aquí la función
recibe `question`.

```python
{
  "inputs": {"question": <str>},
  "expectations": {
    "expected_response": <str>,
    "gold_chunk_ids": [<str>, ...]
  }
}
```

En este dataset reducido cada pregunta tiene un único contexto gold.


In [0]:
eval_rows = (
    spark.table(T_CORPUS)
    .select("question_id", "question", "expected_response", "chunk_id")
    .orderBy("question_id")
    .limit(N_EVAL)
    .collect()
)

eval_data = [
    {
        "inputs": {"question": row["question"]},
        "expectations": {
            "expected_response": row["expected_response"],
            "gold_chunk_ids": [row["chunk_id"]],
        },
    }
    for row in eval_rows
]

assert eval_data, "El dataset de evaluación quedó vacío."
print(f"Casos preparados: {len(eval_data)}")
print(json.dumps(eval_data[0], ensure_ascii=False, indent=2))


## 4. Capa A — scorers determinísticos

Estos scorers no hacen llamadas adicionales al LLM:

- `citation_precision = |citado ∩ gold| / |citado|`;
- `citation_recall = |citado ∩ gold| / |gold|`;
- `citation_f1`: media armónica;
- `retrieval_hit`: separa un fallo de búsqueda de un fallo de citación;
- `answer_token_f1`: proxy gratuito de correctitud al estilo SQuAD.


In [0]:
from mlflow.entities import Feedback
from mlflow.genai.scorers import scorer


def _citation_prf(cited, gold) -> tuple[float, float, float]:
    cited_set = set(cited or [])
    gold_set = set(gold or [])
    intersection = len(cited_set & gold_set)
    precision = intersection / len(cited_set) if cited_set else 0.0
    recall = intersection / len(gold_set) if gold_set else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall > 0
        else 0.0
    )
    return precision, recall, f1


@scorer
def citation_precision(outputs, expectations) -> Feedback:
    cited = (outputs or {}).get("cited_chunk_ids", [])
    gold = (expectations or {}).get("gold_chunk_ids", [])
    precision, _, _ = _citation_prf(cited, gold)
    return Feedback(
        value=precision,
        rationale=(
            f"{len(set(cited) & set(gold))} citas correctas de "
            f"{len(set(cited))} citas emitidas."
        ),
    )


@scorer
def citation_recall(outputs, expectations) -> Feedback:
    cited = (outputs or {}).get("cited_chunk_ids", [])
    gold = (expectations or {}).get("gold_chunk_ids", [])
    _, recall, _ = _citation_prf(cited, gold)
    return Feedback(
        value=recall,
        rationale=(
            f"{len(set(cited) & set(gold))} chunks gold citados de "
            f"{len(set(gold))} requeridos."
        ),
    )


@scorer
def citation_f1(outputs, expectations) -> float:
    cited = (outputs or {}).get("cited_chunk_ids", [])
    gold = (expectations or {}).get("gold_chunk_ids", [])
    return _citation_prf(cited, gold)[2]


@scorer
def retrieval_hit(outputs, expectations) -> Feedback:
    retrieved = set((outputs or {}).get("retrieved_chunk_ids", []))
    gold = set((expectations or {}).get("gold_chunk_ids", []))
    hit = bool(retrieved & gold)
    return Feedback(
        value=1.0 if hit else 0.0,
        rationale=(
            "La evidencia gold apareció en retrieval."
            if hit
            else "Ningún chunk gold fue recuperado: fallo de retrieval."
        ),
    )


@scorer
def answer_token_f1(outputs, expectations) -> float:
    def token_set(text: str) -> set[str]:
        return set(re.findall(r"\w+", (text or "").lower()))

    predicted = token_set((outputs or {}).get("answer", ""))
    expected = token_set((expectations or {}).get("expected_response", ""))
    if not predicted or not expected:
        return 0.0

    overlap = len(predicted & expected)
    if overlap == 0:
        return 0.0

    precision = overlap / len(predicted)
    recall = overlap / len(expected)
    return 2 * precision * recall / (precision + recall)


DETERMINISTIC_SCORERS = [
    citation_precision,
    citation_recall,
    citation_f1,
    retrieval_hit,
    answer_token_f1,
]

print("Scorers:", [s.name for s in DETERMINISTIC_SCORERS])


## 5. Ejecutar `v1` y `v2` como runs separados

Las dos configuraciones mantienen fijo el retriever, `k` y la política. Solo cambia el
prompt, de modo que la comparación prueba una hipótesis interpretable.

Las métricas agregadas quedan en `results.metrics` con sufijo `/mean`, por ejemplo
`citation_f1/mean`.


In [0]:
CONFIGS = [
    {
        "retriever": "ai_search",
        "k": K_BASE,
        "prompt_version": "v1",
        "citation_policy": POLICY_BASE,
    },
    {
        "retriever": "ai_search",
        "k": K_BASE,
        "prompt_version": "v2",
        "citation_policy": POLICY_BASE,
    },
]


def make_agent_fn(config: dict):
    def predict(question: str) -> dict:
        return rag_agent(
            question=question,
            k=config["k"],
            prompt_version=config["prompt_version"],
            citation_policy=config["citation_policy"],
        )

    return predict


def mean_metrics(result) -> dict:
    return {
        key: float(value)
        for key, value in result.metrics.items()
        if key.endswith("/mean") and isinstance(value, (int, float))
    }


results_by_prompt = {}
run_ids = {}
summary_rows = []

for config in CONFIGS:
    run_name = (
        f"eval_vector_k{config['k']}_"
        f"{config['prompt_version']}_{config['citation_policy']}"
    )
    agent_fn = make_agent_fn(config)

    with mlflow.start_run(run_name=run_name) as active_run:
        mlflow.log_params({
            **config,
            "llm_endpoint": LLM_ENDPOINT,
            "n_eval": len(eval_data),
            "layer": "deterministic",
        })
        result = mlflow.genai.evaluate(
            data=eval_data,
            predict_fn=agent_fn,
            scorers=DETERMINISTIC_SCORERS,
        )

    prompt_version = config["prompt_version"]
    results_by_prompt[prompt_version] = result
    run_ids[prompt_version] = active_run.info.run_id
    metrics = mean_metrics(result)

    summary_rows.append({
        "run_id": active_run.info.run_id,
        "layer": "deterministic",
        "status": "OK",
        "retriever": config["retriever"],
        "k": config["k"],
        "prompt_version": prompt_version,
        "citation_policy": config["citation_policy"],
        "llm_endpoint": LLM_ENDPOINT,
        "n_eval": len(eval_data),
        "citation_precision_mean": metrics.get("citation_precision/mean"),
        "citation_recall_mean": metrics.get("citation_recall/mean"),
        "citation_f1_mean": metrics.get("citation_f1/mean"),
        "retrieval_hit_mean": metrics.get("retrieval_hit/mean"),
        "answer_token_f1_mean": metrics.get("answer_token_f1/mean"),
        "retriever_context_present_mean": None,
        "correctness_mean": None,
        "retrieval_groundedness_mean": None,
        "retrieval_relevance_mean": None,
    })

    print(f"✓ {run_name}: {active_run.info.run_id}")


### Comparación de la Capa A


In [0]:
import pandas as pd

comparison_rows = []
for prompt_version, result in results_by_prompt.items():
    row = {"prompt_version": prompt_version}
    row.update(mean_metrics(result))
    comparison_rows.append(row)

comparison_pd = pd.DataFrame(comparison_rows).sort_values("prompt_version")
display(comparison_pd)

print(
    "Busca las columnas citation_f1/mean, retrieval_hit/mean y "
    "answer_token_f1/mean. Retrieval hit indica si el contexto correcto llegó; "
    "citation F1 indica si el agente lo citó correctamente."
)


## 6. Capa B — LLM judges sobre una muestra pequeña

`N_JUDGE = min(10, len(eval_data))`. Cada judge implica una llamada adicional por pregunta,
por eso esta capa se ejecuta una sola vez con la configuración `v2`.

El bloque está protegido con `try/except`: un límite 429 no borra los runs determinísticos
que ya quedaron registrados.

`RetrievalGroundedness` y `RetrievalRelevance` no reciben gold. Leen el contexto directamente
de la lista de `Document` guardada en el span `RETRIEVER`.

Antes de gastar llamadas de judges, este notebook ejecuta una prueba de contrato sin LLM:
comprueba tanto el retorno de `retrieve()` como el span ya persistido. Cada documento debe
tener `page_content` no vacío, `metadata.chunk_id` y `metadata.doc_uri`. Si el esquema no es
válido, la evaluación se detiene con un error explícito en vez de producir silenciosamente
rationales como *the retrieved context is empty*.


In [0]:
def _document_field(document, field: str):
    "Lee Document dataclass, Pydantic o su representación dict persistida."
    if isinstance(document, dict):
        return document.get(field)

    value = getattr(document, field, None)
    if value is not None:
        return value

    for method_name in ("to_dict", "model_dump", "dict"):
        converter = getattr(document, method_name, None)
        if callable(converter):
            converted = converter()
            if isinstance(converted, dict):
                return converted.get(field)
    return None


def _assert_retriever_documents(documents, source: str) -> None:
    assert isinstance(documents, (list, tuple)) and documents, (
        f"{source}: el output RETRIEVER no es una lista no vacía de documentos. "
        f"Valor observado: {documents!r}"
    )

    for position, document in enumerate(documents, start=1):
        if (
            isinstance(document, dict)
            and "chunk_text" in document
            and "page_content" not in document
        ):
            raise AssertionError(
                f"{source}: documento {position} usa el esquema anterior "
                f"list[dict] con claves {sorted(document)}. Actualiza el notebook 55, "
                "reinicia Python y vuelve a ejecutar el notebook 56 desde la primera celda."
            )
        page_content = _document_field(document, "page_content")
        metadata = _document_field(document, "metadata") or {}
        assert isinstance(page_content, str) and page_content.strip(), (
            f"{source}: documento {position} sin page_content. "
            "El judge lo interpretaría como contexto vacío."
        )
        assert isinstance(metadata, dict), (
            f"{source}: metadata inválida en documento {position}: {metadata!r}"
        )
        assert metadata.get("chunk_id"), (
            f"{source}: falta metadata.chunk_id en documento {position}."
        )
        assert metadata.get("doc_uri"), (
            f"{source}: falta metadata.doc_uri en documento {position}."
        )


@mlflow.trace(name="validar_contrato_retriever")
def _retriever_contract_probe(question: str, k: int) -> dict:
    documents = retrieve(question=question, k=k)
    _assert_retriever_documents(documents, "retorno de retrieve()")
    return {"n_documents": len(documents)}


probe_question = eval_data[0]["inputs"]["question"]
_retriever_contract_probe(question=probe_question, k=K_BASE)
probe_trace_id = mlflow.get_last_active_trace_id()
assert probe_trace_id, "MLflow no devolvió trace_id para la prueba del retriever."

probe_trace = None
for attempt in range(5):
    try:
        probe_trace = mlflow.get_trace(probe_trace_id)
        if probe_trace is not None:
            break
    except Exception:
        if attempt == 4:
            raise
    time.sleep(0.5 * (attempt + 1))

assert probe_trace is not None, f"No se pudo leer la traza {probe_trace_id}."
probe_retriever_spans = probe_trace.search_spans(span_type=SpanType.RETRIEVER)
assert probe_retriever_spans, "La prueba no produjo un span RETRIEVER."
persisted_documents = probe_retriever_spans[-1].outputs
_assert_retriever_documents(persisted_documents, "span RETRIEVER persistido")

first_document = persisted_documents[0]
first_text = _document_field(first_document, "page_content")
first_metadata = _document_field(first_document, "metadata") or {}
print("✓ Contrato RETRIEVER válido antes de ejecutar judges")
print(f"  Trace ID       : {probe_trace_id}")
print(f"  Documentos     : {len(persisted_documents)}")
print(f"  Primer chunk_id: {first_metadata.get('chunk_id')}")
print(f"  Caracteres     : {len(first_text)}")
print(f"  Vista previa   : {first_text[:180]!r}")


from mlflow.genai.scorers import (
    Correctness,
    RetrievalGroundedness,
    RetrievalRelevance,
)


@scorer
def retriever_context_present(trace) -> Feedback:
    "Falla de forma visible si el trace del caso no contiene contexto evaluable."
    retriever_spans = trace.search_spans(span_type=SpanType.RETRIEVER)
    if not retriever_spans:
        return Feedback(
            value=0.0,
            rationale="La traza no contiene ningún span RETRIEVER.",
        )

    try:
        documents = retriever_spans[-1].outputs
        _assert_retriever_documents(documents, "trace evaluada")
        total_chars = sum(
            len(_document_field(document, "page_content"))
            for document in documents
        )
        return Feedback(
            value=1.0,
            rationale=(
                f"El último span RETRIEVER contiene {len(documents)} documentos "
                f"y {total_chars} caracteres de contexto."
            ),
        )
    except AssertionError as error:
        return Feedback(value=0.0, rationale=str(error))

N_JUDGE = min(10, len(eval_data))
judge_data = eval_data[:N_JUDGE]
JUDGE_CONFIG = next(c for c in CONFIGS if c["prompt_version"] == "v2")
judge_fn = make_agent_fn(JUDGE_CONFIG)
judge_result = None
judge_run_id = None

try:
    judge_run_name = (
        f"eval_judges_vector_k{JUDGE_CONFIG['k']}_"
        f"{JUDGE_CONFIG['prompt_version']}_{JUDGE_CONFIG['citation_policy']}"
    )
    with mlflow.start_run(run_name=judge_run_name) as judge_run:
        judge_run_id = judge_run.info.run_id
        mlflow.log_params({
            **JUDGE_CONFIG,
            "llm_endpoint": LLM_ENDPOINT,
            "n_eval": N_JUDGE,
            "layer": "llm_judges",
        })
        judge_result = mlflow.genai.evaluate(
            data=judge_data,
            predict_fn=judge_fn,
            scorers=[
                retriever_context_present,
                Correctness(),
                RetrievalGroundedness(),
                RetrievalRelevance(),
            ],
        )

    judge_metrics = mean_metrics(judge_result)
    summary_rows.append({
        "run_id": judge_run_id,
        "layer": "llm_judges",
        "status": "OK",
        "retriever": JUDGE_CONFIG["retriever"],
        "k": JUDGE_CONFIG["k"],
        "prompt_version": JUDGE_CONFIG["prompt_version"],
        "citation_policy": JUDGE_CONFIG["citation_policy"],
        "llm_endpoint": LLM_ENDPOINT,
        "n_eval": N_JUDGE,
        "citation_precision_mean": None,
        "citation_recall_mean": None,
        "citation_f1_mean": None,
        "retrieval_hit_mean": None,
        "answer_token_f1_mean": None,
        "retriever_context_present_mean": judge_metrics.get(
            "retriever_context_present/mean"
        ),
        "correctness_mean": judge_metrics.get("correctness/mean"),
        "retrieval_groundedness_mean": judge_metrics.get(
            "retrieval_groundedness/mean"
        ),
        "retrieval_relevance_mean": judge_metrics.get(
            "retrieval_relevance/mean"
        ),
    })
    print(f"✓ Judges completados. Run: {judge_run_id}")
    display(pd.DataFrame([judge_metrics]))
except Exception as e:
    summary_rows.append({
        "run_id": judge_run_id,
        "layer": "llm_judges",
        "status": f"ERROR: {type(e).__name__}",
        "retriever": JUDGE_CONFIG["retriever"],
        "k": JUDGE_CONFIG["k"],
        "prompt_version": JUDGE_CONFIG["prompt_version"],
        "citation_policy": JUDGE_CONFIG["citation_policy"],
        "llm_endpoint": LLM_ENDPOINT,
        "n_eval": N_JUDGE,
        "citation_precision_mean": None,
        "citation_recall_mean": None,
        "citation_f1_mean": None,
        "retrieval_hit_mean": None,
        "answer_token_f1_mean": None,
        "retriever_context_present_mean": None,
        "correctness_mean": None,
        "retrieval_groundedness_mean": None,
        "retrieval_relevance_mean": None,
    })
    print(
        f"Los judges no terminaron: {type(e).__name__}: {e}\n"
        "La Capa A ya está registrada. Causa habitual: fair-use throttle de Free Edition. "
        "Reintenta más tarde o reduce n_eval."
    )


## 7. Trazas y análisis manual por caso

Primero recuperamos las trazas del run base con `mlflow.search_traces(run_id=...)`.
Después hacemos una corrida manual con la configuración `v2` para clasificar:

- **fallo de retrieval**: el gold nunca llegó; revisar índice, `k` o chunking;
- **fallo de citación**: el gold llegó pero no fue citado; revisar prompt o política;
- **ok_citas**: la evidencia gold fue recuperada y citada.

Esta segunda pasada consume una llamada por caso. Se mantiene pequeña mediante `n_eval`.


In [0]:
TRACE_RUN_ID = run_ids["v1"]
try:
    trace_table = mlflow.search_traces(run_id=TRACE_RUN_ID)
    print(f"Trazas del run v1: {len(trace_table)}")
    display(trace_table.head(10))
except Exception as e:
    print(f"No se pudieron listar trazas por run_id: {type(e).__name__}: {e}")


In [0]:
analysis_config = JUDGE_CONFIG
analysis_fn = make_agent_fn(analysis_config)
case_rows = []

for position, case in enumerate(eval_data, start=1):
    question = case["inputs"]["question"]
    gold = set(case["expectations"]["gold_chunk_ids"])
    try:
        output = analysis_fn(question)
        retrieved = set(output.get("retrieved_chunk_ids", []))
        cited = set(output.get("cited_chunk_ids", []))

        if not (retrieved & gold):
            failure_type = "fallo_retrieval"
        elif not (cited & gold):
            failure_type = "fallo_citacion"
        else:
            failure_type = "ok_citas"

        precision, recall, f1 = _citation_prf(cited, gold)
        case_rows.append({
            "caso": position,
            "question": question,
            "gold_chunk_ids": sorted(gold),
            "retrieved_chunk_ids": sorted(retrieved),
            "cited_chunk_ids": sorted(cited),
            "failure_type": failure_type,
            "citation_precision": precision,
            "citation_recall": recall,
            "citation_f1": f1,
            "answer": output.get("answer"),
            "error": None,
        })
    except Exception as e:
        case_rows.append({
            "caso": position,
            "question": question,
            "gold_chunk_ids": sorted(gold),
            "retrieved_chunk_ids": [],
            "cited_chunk_ids": [],
            "failure_type": "error_llamada",
            "citation_precision": 0.0,
            "citation_recall": 0.0,
            "citation_f1": 0.0,
            "answer": None,
            "error": f"{type(e).__name__}: {e}",
        })
    time.sleep(0.4)

case_pd = pd.DataFrame(case_rows)
display(case_pd.sort_values(["failure_type", "citation_f1", "caso"]))
display(
    case_pd.groupby("failure_type", dropna=False)
    .size()
    .reset_index(name="cantidad")
)


## 8. Persistir el resumen agregado en Delta

MLflow es la fuente rica de runs, trazas, parámetros y rationale. La tabla Delta funciona
como registro redundante, fácil de consultar con SQL y gobernado por Unity Catalog.

Usamos `append` para conservar la historia de distintas clases y usuarios.


In [0]:
from datetime import datetime, timezone

T_EVAL_SUMMARY = f"{CATALOG}.{SCHEMA}.agenteval_eval_summary"
recorded_at_utc = datetime.now(timezone.utc).isoformat()

for row in summary_rows:
    row["experiment_path"] = EXPERIMENT_PATH
    row["experiment_id"] = EXPERIMENT.experiment_id
    row["recorded_at_utc"] = recorded_at_utc
    row["current_user"] = CURRENT_USER

summary_df = spark.createDataFrame(summary_rows)

(
    summary_df.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(T_EVAL_SUMMARY)
)

spark.sql(
    f"""
    ALTER TABLE {T_EVAL_SUMMARY}
    SET TBLPROPERTIES (
      'layer' = 'evaluation',
      'mlflow.experiment_path' = '{EXPERIMENT_PATH}'
    )
    """
)
spark.sql(
    f"COMMENT ON TABLE {T_EVAL_SUMMARY} IS "
    "'Resumen gobernado de evaluaciones RAG registradas en MLflow.'"
)

print(f"Resumen guardado en {T_EVAL_SUMMARY}")
display(
    spark.table(T_EVAL_SUMMARY)
    .orderBy(F.desc("recorded_at_utc"))
    .limit(30)
)


## 9. Comparar y diagnosticar en la UI de MLflow

1. Abre **Experiments** → `/Users/<tu_usuario>/agenteval_rag`.
2. Selecciona los runs `eval_vector_k..._v1_...` y `eval_vector_k..._v2_...`.
3. Pulsa **Compare**.
4. Compara parámetros (`k`, `prompt_version`, `citation_policy`) y métricas con `/mean`.
5. Abre una traza con bajo score.
6. Expande `RETRIEVER`: revisa documentos, ranking y `chunk_id`.
7. Expande `LLM`: revisa prompt, contexto y respuesta.
8. En el run de judges verifica primero que `retriever_context_present/mean = 1.0`.
9. Lee el texto **rationale**, no solo el número. Ya no debe decir que el documento está vacío;
   debe razonar sobre el texto visible dentro de `page_content`.

Interpretación:

- `retrieval_hit` bajo → trabajar en índice, `k` o chunking;
- `retrieval_hit` alto + `citation_f1` bajo → trabajar en prompt/política;
- `citation_f1` alto + `answer_token_f1` bajo → el contexto/cita es correcto, pero la
  respuesta necesita mejorar;
- `retriever_context_present < 1` → contrato de traza roto; no interpretes todavía los judges;
- judge de groundedness bajo con contexto presente → la respuesta afirma algo que los chunks
  no respaldan.

Fin de la serie 52–56.
